In [ ]:
import pandas as pd

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
dataset = pd.read_csv("estado_nuticional_sao_paulo.csv")

dataset.head()

In [ ]:
dataset.tail()

In [ ]:
dataset.info()

Importamos uma base com informações da situação nutricional das pessoas da cidade de São Paulo/SP no período de Janeiro de 2022 à Dezembro de 2023 para treinar nosso modelo.

O objetivo é treinar o modelo para ele analisar dados como idade, altura, peso, etc... e definir em qual estado nutricional estão as pessoas, Magreza, Sobrepeso, Obesidade, etc...

Avaliando os primeiros e últimos registros da base de dados, podemos notar a preseça de algumas colunas irrelevantes para o treinamento do modelo, como códigos de sistemas e datas. 

Além disso, muitas colunas aparecem totalmente nulas e outras majoritariamente nulas, sendo que aplicar uma mediana não seria adequado porque são dados determiníticos, como 'DS_POVO_COMUNIDADE', que é a comunidade aonde a pessoa reside. Não tem uma técnica para "adivinhar" em qual comunidade a pessoa reside.

Abaixo temos a relação de colunas que vamos remover da base para trabalharmos somente com dados relevantes para o treinamento.

"ST_PARTICIPA_ANDI", # Irrelevante: Se o municipio participa do programa ANDI  
"CO_POVO_COMUNIDADE", # A maioria é nulo  
"DS_IMC_PRE_GESTACIONAL", # Todos nulos  
"CO_ESCOLARIDADE", # A maioria é nulo  
"DS_ESCOLARIDADE", # A maioria é SEM INFORMAÇÃO  
"CO_ACOMPANHAMENTO", # Código irrelevante  
"CO_PESSOA_SISVAN", # Código irrelevante
"CO_MUNICIPIO_IBGE", # Todos são SÃO PAULO  
"SG_UF", # Todos são SÃO PAULO  
"NO_MUNICIPIO", # Todos são SÃO PAULO  
"CO_CNES", # Código irrelevante  
"DS_POVO_COMUNIDADE", # A maioria é NÃO INFORMADO  
"NU_COMPETENCIA", # Ano/Mês do acompanhamento, irrelevante neste caso  
"CO_SISTEMA_ORIGEM_ACOMP", # Código irrelevante  
"SISTEMA_ORIGEM_ACOMP", # Informação irrelevante  
"DT_ACOMPANHAMENTO", # Data, irrelevante neste caso  
"PESO X IDADE", # Somente para as idades de 0 a 10 anos  
"CRI. ALTURA X IDADE", # Somente de 0 a 10 anos  
"ADO. ALTURA X IDADE", # Somente de 10 a menos de 20 anos  
"PESO X ALTURA", # Tem na coluna CRI. IMC X IDADE  

OBS.: Qualquer dúvida sobre os dados, vide o arquivo 'Dicionário_de_Dados_Estado_Nutricional.pdf' que está inclúso neste projeto.

In [ ]:
clean_dataset = dataset.drop(columns=[
                                "ST_PARTICIPA_ANDI", # Irrelevante: Se o municipio participa do programa ANDI
                                "CO_POVO_COMUNIDADE", # A maioria é nulo
                                "DS_IMC_PRE_GESTACIONAL", # Todos nulos
                                "CO_ESCOLARIDADE", # A maioria é nulo
                                "DS_ESCOLARIDADE", # A maioria é SEM INFORMAÇÃO
                                "CO_ACOMPANHAMENTO", # Código irrelevante
                                "CO_PESSOA_SISVAN", # Código irrelevante
                                "CO_MUNICIPIO_IBGE", # Todos são SÃO PAULO
                                "SG_UF", # Todos são SÃO PAULO
                                "NO_MUNICIPIO", # Todos são SÃO PAULO
                                "CO_CNES", # Código irrelevante
                                "DS_POVO_COMUNIDADE", # A maioria é NÃO INFORMADO
                                "NU_COMPETENCIA", # Ano/Mês do acompanhamento, irrelevante neste caso
                                "CO_SISTEMA_ORIGEM_ACOMP", # Código irrelevante
                                "SISTEMA_ORIGEM_ACOMP", # Informação irrelevante
                                "DT_ACOMPANHAMENTO", # Data, irrelevante neste caso
                                "PESO X IDADE", # Somente para as idades de 0 a 10 anos
                                "CRI. ALTURA X IDADE", # Somente de 0 a 10 anos
                                "ADO. ALTURA X IDADE", # Somente de 10 a menos de 20 anos
                                "PESO X ALTURA", # Tem na coluna CRI. IMC X IDADE
                                ])
clean_dataset.info()

In [ ]:
clean_dataset.head()

Com base limpa, podemos ver que as colunas restantes conetem bem menos informações nulas e nenhuma irrelevante para o treinamento.

Abaixo temos a relação de colunas variáveis, que são os dados a serem analisados para chegar na conclusão final que é o estado nutricional da pessoa.
Note que temos todos os dados preenchidos sem nulos em nenhum registro.

| # | Column | Non-Null Count | Dtype |
|---| :--- | :--- | :--- |
| 0 | NU_IDADE_ANO | 1.582.936 | int64 |
| 1 | NU_FASE_VIDA | 1.582.936 | float64 |
| 2 | DS_FASE_VIDA | 1.582.936 | str |
| 3 | SG_SEXO | 1.582.936 | str |
| 4 | CO_RACA_COR | 1.582.936 | int64 |
| 5 | DS_RACA_COR | 1.582.936 | str |
| 6 | NU_PESO | 1.582.936 | str |
| 7 | NU_ALTURA | 1.582.936 | str |
| 8 | DS_IMC | 1.582.936 | str |

 Os dados como idade, fase de vida, sexo, raça/cor, peso, altura e IMC serão analisados para chegar na conclusão final, nosso alvo (target).

 Anaisando os dados atuais, após a limpeza, notamos que nosso target está distribuído em 5 colunas diferentes que se diferenciam pela faixa etária dos indivíduos.

| # | Column | Non-Null Count | Dtype |
|---| :--- | :--- | :--- |
| 9 | CRI. IMC X IDADE | 347.088 | str |
| 10 | ADO. IMC X IDADE | 190.113 | str |
| 11 | CO_ESTADO_NUTRI_ADULTO | 725.655 | str |
| 12 | CO_ESTADO_NUTRI_IDOSO | 309.080 | str |
| 13 | CO_ESTADO_NUTRI_IMC_SEMGEST | 38.933 | str |

A coluna 'CRI. IMC X IDADE' traz informações de pessoas entre 0 a 10 anos, aonde os valores são: 'Magreza acentuada’, ‘Magreza’, ‘Eutrofia', 'Riscode sobrepeso', 'Sobrepeso', 'Obesidade'. Nas demais pessoas fora desta faixa de idade, essa coluna tera a informação nula.

Ja a coluna 'ADO. IMC X IDADE' tras informações de pessoas entre 10 e 20 anos, aonde os valores são: Magreza acentuada', 'Magreza', 'Eutrofia', 'Risco de sobrepeso','Sobrepeso', 'Obesidade'. As pessoas fora desta faixa de idade terão essa coluna nula.

A coluna 'CO_ESTADO_NUTRI_ADULTO' tras informações de pessoas entre 20 e 60 anos, aonde os valores são: 'Baixo peso', 'Adequado ou Eutrófico', 'Sobrepeso', 'Obesidade Grau I','Obesidade Grau II', 'Obesidade Grau III'. Note aqui o resultado é um pouco diferente, a obesidade se divide em 3 graus. Isso porque só é possível medir com essa precisão em pessoas nesta faixa etária.

A colina 'CO_ESTADO_NUTRI_IDOSO' tras informações de pessoas com 60 anos ou mais, aonde os valores são: 'Baixo peso', 'Adequado ou Eutrófico','Sobrepeso'

Agora vamos analisar se todos os 1.582.936 registros possuem alguma das informações alvo (target) sem a informação de gestantes (CO_ESTADO_NUTRI_IMC_SEMGEST).

In [ ]:
filtro = (
    clean_dataset['CRI. IMC X IDADE'].isna() & 
    clean_dataset['ADO. IMC X IDADE'].isna() & 
    clean_dataset['CO_ESTADO_NUTRI_ADULTO'].isna() & 
    clean_dataset['CO_ESTADO_NUTRI_IDOSO'].isna()
)

print(f"Toal sem NENHUM target: {len(clean_dataset[filtro])}")

Ficamos com 11000 registros sem o target.

Agora vamos incluir a informação de gestantes para ver se conseguimos mais alguns registros com o target.

In [ ]:
filtro = (
    clean_dataset['CRI. IMC X IDADE'].isna() & 
    clean_dataset['ADO. IMC X IDADE'].isna() & 
    clean_dataset['CO_ESTADO_NUTRI_ADULTO'].isna() & 
    clean_dataset['CO_ESTADO_NUTRI_IDOSO'].isna() &
    clean_dataset['CO_ESTADO_NUTRI_IMC_SEMGEST'].isna()
)

print(f"Toal sem NENHUM target: {len(clean_dataset[filtro])}")

Com esta coluna conseguimos diminuir os registros sem target para 10650 (350 que não tinham nenum dos 4 targets foram classificados com a informação de 'CO_ESTADO_NUTRI_IMC_SEMGEST').

Vamos utilizar esta coluna para classificar os indivíduos que não se enquadram nas ontras colunas.

### Classificação do Estado Nutricional por Grupo (Targets)

*   **CRI. IMC X IDADE** (0 a 10 anos)
    *   **Valores:** `Magreza acentuada`, `Magreza`, `Eutrofia`, `Risco de sobrepeso`, `Sobrepeso`, `Obesidade`.
*   **ADO. IMC X IDADE** (10 a menos de 20 anos)
    *   **Valores:** `Magreza acentuada`, `Magreza`, `Eutrofia`, `Risco de sobrepeso`, `Sobrepeso`, `Obesidade`.
*   **CO_ESTADO_NUTRI_ADULTO** (20 a 60 anos)
    *   **Valores:** `Baixo peso`, `Adequado ou Eutrófico`, `Sobrepeso`, `Obesidade Grau I`, `Obesidade Grau II`, `Obesidade Grau III`.
*   **CO_ESTADO_NUTRI_IDOSO** (60 anos ou mais)
    *   **Valores:** `Baixo peso`, `Adequado ou Eutrófico`, `Sobrepeso`.
*   **CO_ESTADO_NUTRI_IMC_SEMGEST** (Gestantes de 10 a 60 anos)
    *   **Valores:** `Baixo peso`, `Adequado ou Eutrófico`, `Sobrepeso`, `Obesidade`.
    *   *Nota: Aplicar caso não conste em nenhuma das categorias acima.*

Com esta análise, podemos juntar os targets em uma única coluna, escolhendo aquelas que cada registro possui.

In [ ]:
'''
# Metódo não performático
clean_dataset["ESTADO_NUTRI"] =  clean_dataset['CRI. IMC X IDADE'] \
    .combine_first(clean_dataset['ADO. IMC X IDADE']) \
    .combine_first(clean_dataset['CO_ESTADO_NUTRI_ADULTO']) \
    .combine_first(clean_dataset['CO_ESTADO_NUTRI_IDOSO']) \
    .combine_first(clean_dataset['CO_ESTADO_NUTRI_IMC_SEMGEST'])
'''

# Método meio performático
colunas = [
    'CRI. IMC X IDADE', 
    'ADO. IMC X IDADE', 
    'CO_ESTADO_NUTRI_ADULTO', 
    'CO_ESTADO_NUTRI_IDOSO', 
    'CO_ESTADO_NUTRI_IMC_SEMGEST'
]

# Ele "empurra" o primeiro valor encontrado da esquerda para a direita
clean_dataset['ESTADO_NUTRI'] = clean_dataset[colunas].bfill(axis=1).iloc[:, 0]

clean_dataset.info()
# OBS.: Existe um método mais performárico (e complexo) com numpy, 
# mas para esta base de dados, o 'bfill' vai atender bem, demorou uns 50s para realizar esta operação.

Com a nova coluna 'ESTADO_NUTRI' consolidada, podemos ver que 1572286 dos 1582936 registros foram classificados com sucesso.

Com a nova coluna target estabelecida, podemos remover as demais que foram usadas para gerar ela.

In [ ]:
clean_target_dataset = clean_dataset.drop(columns=[
                                                    "CRI. IMC X IDADE",
                                                    "ADO. IMC X IDADE",
                                                    "CO_ESTADO_NUTRI_ADULTO",
                                                    "CO_ESTADO_NUTRI_IDOSO",
                                                    "CO_ESTADO_NUTRI_IMC_SEMGEST"
                                                    ])

clean_target_dataset.info()

In [ ]:
clean_target_dataset.head()

Com a base quase limpa e com um único target estabelecido, vamos ver quantos e quais registros não conseguiram nenhuma classificação.

In [ ]:
print(f"Quantidade de registros SEM classificação(ESTADO_NUTRI): {len(clean_target_dataset[clean_target_dataset["ESTADO_NUTRI"].isna()])}")

In [ ]:
clean_target_dataset[clean_target_dataset["ESTADO_NUTRI"].isna()][:20]

Como a base possui mais de 1.5 milhões de registro, remover esta pequena quantidade de 10650 sem classificação não vai prejudicar o trenamento do modelo.

Vamos faser isso e analiar os dados novamente.

In [ ]:
clean_target_dataset.dropna(subset=['ESTADO_NUTRI'], inplace=True)

clean_target_dataset.info()

In [ ]:
clean_target_dataset[:20]

Agora sim, com a base limpa, aonde temos todos os dados com valores e alvos informados, vamos trabalhar na normalização dos dados, tranformando o que texto em números.

Para isso, primeiro vamos removes os códigos existentes (NU_FASE_VIDA e CO_RACA_COR) e deixar somente os textos (DS_FASE_VIDA e DS_RACA_COR) para aplicarmos a alteração via recursos da Sk Learn.

In [ ]:
clean_target_dataset.drop(columns=[
                                    "NU_FASE_VIDA",
                                    "CO_RACA_COR"
                                    ], inplace=True)

clean_target_dataset[:20]

### Analisando as colunas referentes à faixa etária de 0 a 10 anos

As colunas alvo desta faixa etária (conforme descrito na documentação) apresentam quantidade de dados diferentes. Vamos avaliar a distribição da idade entre as colunas para entendimento dos dados.

19  PESO X IDADE             347146 non-null   str    
20  PESO X ALTURA            236479 non-null   str    
21  CRI. ALTURA X IDADE      347154 non-null   str    
22  CRI. IMC X IDADE         347088 non-null   str  

In [ ]:
df_peso_idade = dataset.loc[dataset['PESO X IDADE'].notna(), ['NU_IDADE_ANO']]
df_peso_altura = dataset.loc[dataset['PESO X ALTURA'].notna(), ['NU_IDADE_ANO']]
df_cri_altura_idade = dataset.loc[dataset['CRI. ALTURA X IDADE'].notna(), ['NU_IDADE_ANO']]
df_cri_imc_idade = dataset.loc[dataset['CRI. IMC X IDADE'].notna(), ['NU_IDADE_ANO']]

In [ ]:
df_peso_idade.value_counts()

In [ ]:
df_peso_altura.value_counts()

In [ ]:
df_cri_altura_idade.value_counts()

In [ ]:
df_cri_imc_idade.value_counts()

Avaliando-se a faixa etária coberta por cada coluna, nota-se que a coluna "PESO X ALTURA", que de acordo com o dicionário de dados deveria cobrir a mesma faixa etária das demais, tem menos dados. Vamos analisar se estes dados aparecem em conjunto com as outras colunas nos mesmos registros ou se estão em amostras separadas.

In [ ]:
df_all_filled = dataset.loc[dataset['PESO X ALTURA'].notna() & dataset['PESO X IDADE'].notna() & dataset['CRI. ALTURA X IDADE'].notna() & dataset['CRI. IMC X IDADE'].notna(), ['NU_IDADE_ANO'] ]
df_1_null = dataset.loc[dataset['PESO X ALTURA'].notna() & (dataset['PESO X IDADE'].isna() | dataset['CRI. ALTURA X IDADE'].isna() | dataset['CRI. IMC X IDADE'].isna()), ['NU_IDADE_ANO'] ]

In [ ]:
print(len(df_all_filled))
print(len(df_1_null))

Baseado nos números de dados preenchidos, a coluna PESO x ALTURA apresenta um dado específico para crianças de 0 a 6 anos e está preenchida em conjunto com as demais colunas da faixa etária de 0 a 8 anos.